In [6]:
import copy
import itertools
import random
import math
import networkx as nx
import pandas as pd
import numpy as np
import time
import pickle
from collections import Counter
import matplotlib.pyplot as plt
from collections import defaultdict
from functools import reduce
import gc
path='wiki10'
with open("./dataset/"+path+'/'+"sh.pkl", "rb") as f:
    sh=pickle.load(f)
print(len(sh))

216345


In [24]:
k = 120  # 【在此处设置】你想要随机选择的社区数量

In [25]:
import sys
import copy
import itertools
import random
import math
import networkx as nx
import pandas as pd
import numpy as np
import time
import pickle
from collections import Counter
import matplotlib.pyplot as plt
from collections import defaultdict
from functools import reduce
import gc
import heapq

ccs=0.1
path='gnutella'
datasets=[]
f=open("./dataset/"+path+'/'+path+".txt")
data=f.read()
f.close()
rows=data.split('\n')
for row in rows:
    split_row=row.split(' ')
    name=(int(split_row[0]),int(split_row[1]),0.01)
    datasets.append(name)

G=nx.DiGraph()
G.add_weighted_edges_from(datasets)   #根据数据集创建有向图

del data,rows,datasets
gc.collect()

node_list=list(G.nodes)
nodenum=sorted(node_list,reverse=True)[0]+1

total_nodes=len(G.nodes)#总节点数
total_edges=len(G.edges)#总边数
print('total_nodes:',total_nodes, 'total_edges:',total_edges)

# --- 修改开始 ---


print(f"Plan to select {k} random communities.")

# 1. 第一步：读取文件并统计所有出现的社区
raw_community_dict = {}
all_communities_pool = set()

with open("./dataset/"+path+'/'+path+"_mem.txt", 'r') as f:
    for line in f:
        data = line.strip().split()
        node_id = int(data[0])
        group_ids = data[1:]
        
        # 暂时存入原始字典
        raw_community_dict[node_id] = group_ids
        # 收集所有社区ID用于抽样
        all_communities_pool.update(group_ids)

# 2. 第二步：从所有社区中随机抽取 k 个
if len(all_communities_pool) <= k:
    selected_communities = list(all_communities_pool)
    print(f"Warning: k is larger than total communities. Selected all {len(selected_communities)} communities.")
else:
    selected_communities = list(all_communities_pool)[:k]

print(f"Successfully selected {len(selected_communities)} communities.")

# 3. 第三步：更新图节点，只保留属于 selected_communities 的社区
community_set = set() # 用于最后统计实际用到的社区

# 遍历图中的所有节点进行初始化和赋值
for node in G.nodes():
    # 默认为空列表（即没有社区）
    G.nodes[node]['community'] = []
    
    # 如果该节点在原始文件中有记录
    if node in raw_community_dict:
        # 核心逻辑：取交集。只保留那些“被选中”的社区ID
        # list comprehension: 遍历节点原有的社区，只有在 selected_communities 里的才保留
        filtered_groups = [gid for gid in raw_community_dict[node] if gid in selected_communities]
        
        if filtered_groups:
            G.nodes[node]['community'] = filtered_groups
            community_set.update(filtered_groups)

no_community_num = sum(1 for n in G.nodes() if len(G.nodes[n]['community']) == 0)

print(f"没有社区的节点数量: {no_community_num}")
print(f"占比: {no_community_num / total_nodes:.2%}") # 顺便看下比例

# --- 修改结束 ---

# 下面是原本的边权重计算逻辑（保持不变，但因为上面确保了每个节点都有 community 属性，所以不会报错）
for u, v, data in G.edges(data=True):
    G[u][v]['b_uv'] = G[u][v]['weight']
    # 注意：这里引用了 G.nodes[v]['community']，上面必须确保所有节点都被初始化过
    if len(G.nodes[v]['community']) == 0: 
        G[u][v]['q_uv'] = 0
    else:
        G[u][v]['q_uv'] = ccs * (1 - G[u][v]['b_uv']) / len(G.nodes[v]['community'])

community_list = list(community_set)
print('community_list len:', len(community_list))

def RRS(g=G,t=node_list):# rr-seed
    seed = random.choice(t)
    result_list = []
    result_list.append(seed)
    RRS=[]
    RRS.append(seed)
    # 保存激活的状态
    checked = np.zeros(nodenum,dtype=bool) 
    for node in result_list:
        checked[node] = 1
    # 当前节点不为空，进行影响
    while len(result_list) != 0:
        current_node = result_list.pop(0)
        for nbr in g.predecessors(current_node): # 得到当前节点的邻居节点
            if checked[nbr] == 0:
                wt = g.get_edge_data(nbr,current_node)
                if random.uniform(0,1) < wt['weight'] :
                    result_list.append(nbr)
                    checked[nbr] = 1
                    RRS.append(nbr)
    return RRS

def generate_1_hop_vertex_cover(G):
    # 创建图的副本，避免修改原图
    graph = G.copy()
    vertex_cover = set()
    
    # 当图中还有边时继续迭代
    while graph.edges():
        # 随机选择一条边
        edge = random.choice(list(graph.edges()))
        
        # 随机选择边的一个顶点
        vertex = random.choice(edge)
        
        # 将顶点加入覆盖集合
        vertex_cover.add(vertex)
        
        # 从图中删除该顶点及其所有关联边
        graph.remove_node(vertex)
    
    return vertex_cover
def generate_2_hop_vertex_cover(G):
    def two_hop_neighbors(G, v):
        # 获取顶点v的邻居
        neighbors_v = set(G.neighbors(v))

        # 获取邻居的邻居
        two_hop_neigh = set()
        for neighbor in neighbors_v:
            two_hop_neigh.update(G.neighbors(neighbor))

        # 两跳的邻居包括邻居的邻居，排除v自己和v的直接邻居
        two_hop_neigh -= neighbors_v
        two_hop_neigh.discard(v)  # 确保不包含v本身

        return two_hop_neigh
    graph=G.copy()
    sh = []
    flag=1
    while(1):
        out_degrees = graph.out_degree()
        max_node = max(out_degrees, key=lambda x: x[1])  # (节点, 出度)
        node=max_node[0]
        visited=two_hop_neighbors(graph, node)
        if(len(visited)==0):
            break
        graph.remove_node(node)
        sh.append(node)
        sh.extend(visited)
        graph.remove_nodes_from(visited)
    print(len(graph.nodes()),len(graph.edges()))
    return sh

def update_weight(g,F):
    def calculate_h_uv(Fv, F, b_uv, q_uv):# 计算 h_uv(Fv, F)
        overlap = len(set(Fv) & set(F))  # 计算 Fv 和 F 的交集大小
        if(q_uv==0): max_increase=0
        else:max_increase = (1 - b_uv) / q_uv  # 根据公式1限制激活概率的上限
        return min(max_increase, overlap)
    G=g.copy()
    # 更新图中每条边的激活概率
    for u, v, data in G.edges(data=True):
        b_uv = data['b_uv']  # 获取基础激活概率
        q_uv = data['q_uv']  # 获取边的增益系数
        F_v = G.nodes[v]['community']  # 获取目标节点 v 的社区集合
        # 根据公式计算新的激活概率
        h_uv_value = calculate_h_uv(F_v, F, b_uv, q_uv)
        new_p_uv = b_uv + q_uv * h_uv_value
        # 更新边的激活概率
        if(new_p_uv!=G[u][v]['weight']):
            G[u][v]['weight'] = new_p_uv
    return G

def find_two_hop_paths_with_max_probability(graph, start): #p1 
    # 存储每个目标节点的前两条最大概率路径
    paths = defaultdict(list)
    
    # 深度优先搜索DFS来查找两跳路径
    def dfs(node, path, prob, depth):
        if depth == 2:
            target = path[-1]
            # 如果两跳路径的目标已经存在，则将新路径插入
            paths[target].append((path, prob))
            return
        
        if node in graph:
            for neighbor in graph.successors(node):
                # 获取当前边的权重（概率）
                edge_prob = graph[node][neighbor]['weight']
                dfs(neighbor, path + [neighbor], prob * edge_prob, depth + 1)
    
    # 查找直接的两跳路径
    for target in graph.successors(start):
        prob = graph[start][target]['weight']
        paths[target].append(([start, target], prob))
    
    # 开始DFS遍历图
    dfs(start, [start], 1, 0)
    
    # 对每个目标节点按概率排序，并仅保留概率最大的两条路径
    result = {}
    for target in paths:
        # 按照路径的总概率排序
        sorted_paths = sorted(paths[target], key=lambda x: x[1], reverse=True)
        result[target] = sorted_paths[:2]  # 取概率最大的两条路径
    
    return result
def find_reverse_two_hop_paths_with_max_probability(graph, start):
    # 创建反向图
    reverse_graph = graph.reverse()  # 反向图
    
    # 存储每个目标节点的前两条最大概率路径
    paths = defaultdict(list)
    
    # 深度优先搜索DFS来查找反向两跳路径
    def dfs(node, path, prob, depth):
        if depth == 2:
            target = path[-1]
            # 如果反向两跳路径的目标已经存在，则将新路径插入
            paths[target].append((path, prob))
            return
        
        if node in reverse_graph:
            for neighbor in reverse_graph.successors(node):
                # 获取当前边的权重（概率）
                edge_prob = reverse_graph[node][neighbor]['weight']  # 默认权重为1
                dfs(neighbor, path + [neighbor], prob * edge_prob, depth + 1)
    
    # 查找直接的反向两跳路径
    for target in reverse_graph.successors(start):
        prob = reverse_graph[start][target]['weight']
        paths[target].append(([start, target], prob))
    
    # 开始DFS遍历反向图
    dfs(start, [start], 1, 0)
    
    # 对每个目标节点按概率排序，并仅保留概率最大的两条路径
    result = {}
    for target in paths:
        # 按照路径的总概率排序
        sorted_paths = sorted(paths[target], key=lambda x: x[1], reverse=True)
        result[target] = sorted_paths[:2]  # 取概率最大的两条路径
    
    return result

def pf_path(result,s,path):
    path[s]={}
    for target, paths in result.items():
        if len(paths)==1: path[s][target]=[paths[0][0]]
        else:
            path[s][target]=[paths[0][0],paths[1][0]]
    return path

def get_forward_probability(G,path_list):
    if len(path_list)==2:
        return G[path_list[0]][path_list[1]]['weight']
    elif len(path_list)==3:
        return G[path_list[0]][path_list[1]]['weight']*G[path_list[1]][path_list[2]]['weight']
    else:
        print(path_list,"get_forward_probability有误！")
        return None

def get_reverse_probability(G,path_list):
    if len(path_list)==2:
        return G[path_list[1]][path_list[0]]['weight']
    elif len(path_list)==3:
        return G[path_list[2]][path_list[1]]['weight']*G[path_list[1]][path_list[0]]['weight']
    else:
        print(path_list,"get_reverse_probability有误！")
        return None

#计算IMM
def RRS_ori(g=G,t=node_list):#最原始的rr-seed 
    seed = random.choice(t)
    result_list = []
    result_list.append(seed)
    RRS=[]
    RRS.append(seed)
    # 保存激活的状态
    checked = np.zeros(nodenum,dtype=bool) 
    for node in result_list:
        checked[node] = 1
    # 当前节点不为空，进行影响
    while len(result_list) != 0:
        current_node = result_list.pop(0)
        for nbr in g.predecessors(current_node): # 得到当前节点的邻居节点
            if checked[nbr] == 0:
                wt = g.get_edge_data(nbr,current_node)
                if random.uniform(0,1) < wt['weight'] :
                    result_list.append(nbr)
                    checked[nbr] = 1
                    RRS.append(nbr)
    return RRS
def combination(n, k):
    if k < 0 or k > n:
        return 0
    # 利用对称性，C(n, k) == C(n, n-k)
    if k > n - k:
        k = n - k
    result = 1
    for i in range(1, k + 1):
        result = result * (n - i + 1) // i
    return result
def lnc(n,k):
    return math.log(combination(n, k))
def nodeselect(RR,k):#对RRSet作种子选择
    s=time.time()
    R=RR.copy()
    js=0
    SEED=[]
    for _ in range(k):
        flat_map = [item for subset in R for item in subset]
        if(len(flat_map))==0:
            return SEED,0
        seed = Counter(flat_map).most_common()[0][0]
        js=js+Counter(flat_map).most_common()[0][1]
        SEED.append(seed)
        R = [rrs for rrs in R if seed not in rrs]
    return SEED,js
def IMM_RRS(G,k,eps,ell,node_list=node_list,n=total_nodes):
    s = time.time()
    # 参数计算（保持不变）
    ell=ell*(1+math.log(2)/math.log(n))
    eps1=math.sqrt(2) * eps
    alpha = math.sqrt(ell * math.log(n) + math.log(2))
    beta = math.sqrt((1.0 - 1.0 / math.exp(1)) * (lnc(n,k) + alpha * alpha))
    lamba1 = (2*n*(1+1/3*eps1) * (lnc(n,k)+ell*math.log(n)+math.log(math.log(n)/math.log(2))))/(eps1*eps1)
    lamba2 = 2 * n * pow(((1.0 - 1.0 / math.exp(1)) * alpha + beta),2) / (eps*eps)
    LB=1
    # 初始化映射（b直接存储RR集，a和cover_counts实时更新）
    a = defaultdict(list)  # a[node] = [rrset_idx1, ...]
    b = []  # b[rr_idx] = [node1, ...]（替代原R）
    cover_counts = defaultdict(int)  # 节点覆盖计数
    
    len_b=0
    # 校准阶段
    for i in range(1, int(math.log(n-1) / math.log(2)) + 1):
        x = n / pow(2, i)
        ti = int(lamba1 / x)
        # 生成RR集并实时更新映射
        
        while len_b < ti:
            rrset = RRS_ori(G, node_list)
            b.append(rrset)
            for node in rrset:
                a[node].append(len_b)
                cover_counts[node] += 1
            len_b=len(b)
        
        # 无需复制，直接传递原始映射（函数内部不会修改它们）
        remaining_rrs = set(range(len(b)))
        SEED, js = optimized_nodeselect(k, a, b, cover_counts, remaining_rrs)
        
        if n * js / len(b) >= (1 + eps1) * x:
            LB = n * js / ((1 + eps1) * len(b))
            break
    
    # 最终采样阶段
    theta = int(lamba2 / LB)
    print(f"len(RR): {theta}")
    
    # 继续生成RR集至目标数量
    
    while len_b < theta:
        rrset = RRS_ori(G, node_list)
        b.append(rrset)
        for node in rrset:
            a[node].append(len_b)
            cover_counts[node] += 1
        len_b=len(b)
    
    # 最终种子选择（同样无需复制）
    remaining_rrs = set(range(len(b)))
    seed, js = optimized_nodeselect(k, a, b, cover_counts, remaining_rrs)
    
    print(f"IMM_RRS COST TIME: {time.time() - s:.2f}s")
    return seed

def optimized_nodeselect(k, a, b, cover_counts, remaining_rrs):
    """无需复制原始映射，仅在内部维护临时状态"""
    s = time.time()
    SEED = []
    total_covered = 0
    
    # 仅复制需要修改的状态（覆盖计数和剩余RR集）
    current_cover = cover_counts.copy()  # 复制需要修改的计数
    current_remaining = remaining_rrs.copy()  # 复制需要修改的集合
    
    for _ in range(k):
        if not current_remaining or not current_cover:
            break
        
        # 选择覆盖最多的节点
        seed = max(current_cover, key=lambda x: current_cover[x])
        SEED.append(seed)
        
        # 找到覆盖的RR集
        covered_rrs = [rr_idx for rr_idx in a[seed] if rr_idx in current_remaining]
        total_covered += len(covered_rrs)
        
        # 更新剩余RR集
        for rr_idx in covered_rrs:
            current_remaining.discard(rr_idx)
        
        # 更新其他节点的覆盖计数
        for rr_idx in covered_rrs:
            for node in b[rr_idx]:
                if node != seed and node in current_cover:
                    current_cover[node] -= 1
                    if current_cover[node] == 0:
                        del current_cover[node]
        
        # 移除已选节点
        del current_cover[seed]
    
    #print(f"节点选择耗时: {time.time() - s:.2f}秒")
    return SEED, total_covered
def calculate_two_hop_probability(graph, start_node):
    hop_probabilities = {}
    
    # 计算第一跳的激活概率
    first_hop_probs = {}
    for neighbor in graph.successors(start_node):  # 只考虑a的邻居
        # 计算第一跳传播概率
        hop_probabilities[neighbor] = graph[start_node][neighbor]['weight']  # 保存第一跳的概率
    # 计算第二跳的激活概率
    for node in graph.successors(start_node):
        for second_node in graph.successors(node):  # 第二跳从node的邻居开始
            # 计算第二跳的传播概率
            if(second_node not in hop_probabilities.keys()):hop_probabilities[second_node]=graph[node][second_node]['weight']*graph[start_node][node]['weight']
            else:
                hop_probabilities[second_node]=1-(1-hop_probabilities[second_node])*(1-graph[node][second_node]['weight']*graph[start_node][node]['weight']) 
    return hop_probabilities

with open("./dataset/"+path+'/'+"sh.pkl", "rb") as f:
    sh=pickle.load(f)

t=time.time()
graph={}
for i in community_list:
    graph[i]=update_weight(G,i)
    
print('gragh:',time.time()-t)

t=time.time()
table={}

for node in sh:
    table[node]={}
    G_probability={}
    G_2hop=calculate_two_hop_probability(G, node)
    temp_dict={}
    for i in community_list:
        G_probability[i]=calculate_two_hop_probability(graph[i], node)
    for key,value in G_2hop.items():
        x0=G_2hop[key]
        for i in community_list:
            temp_dict[i]=G_probability[i][key]
        table[node][key]=(x0,temp_dict)

with open("./dataset/"+path+'/'+"table111.pkl", "wb") as f:
    pickle.dump(table, f)  # 将数据序列化后写入文件
print('table cost time:',time.time()-t)
base_size = sys.getsizeof(table)
print(f"字典If本身占用内存: {base_size} 字节")
print(len(community_list))

total_nodes: 10876 total_edges: 39994
Plan to select 120 random communities.
Successfully selected 120 communities.
没有社区的节点数量: 2125
占比: 19.54%
community_list len: 120
gragh: 21.187364101409912
table cost time: 25.309556007385254
字典If本身占用内存: 295000 字节
120


In [19]:
print(len(community_list))

30


In [13]:
with open("./dataset/"+path+'/'+"If.pkl", "rb") as f:
    If=pickle.load(f)   
base_size = sys.getsizeof(If)
print(f"字典If本身占用内存: {base_size} 字节")
print(len(community_list))

字典If本身占用内存: 295000 字节
120


In [ ]:
295000 ,